# 01. POS Feature Engineering

**출력**:  
-  (NPD 30일 판매·프로모션 피처)  

매칭·진단 작업은  참고.

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
from pathlib import Path
from collections import Counter

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

BASE_DIR = Path(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

# Part 1 입력
IP_MASTER_PATH = BASE_DIR / 'data' / 'processed' / 'ip_master_dataset.parquet'
TREND_KW_PATH  = BASE_DIR / 'data' / 'processed' / 'trend_keywords_processed.parquet'

# Part 2 입력
POS_PATH = BASE_DIR / 'data' / 'processed' / 'POS 전처리 최종' / 'pos_data_food_final_상품단위변환전.parquet'
B4_PATH  = BASE_DIR / 'data' / 'processed' / 'B4_ITEM_DV_INFO_filtered.parquet'
B5_PATH  = BASE_DIR / 'data' / 'processed' / 'B5_MNM_DATA.parquet'
OUT_PATH = BASE_DIR / 'data' / 'processed' / 'pos_product_features.parquet'
OUT_CSV  = BASE_DIR / 'data' / 'processed' / 'pos_product_features.csv'


import re
import difflib
from collections import defaultdict

# Block 5 / Phase 5-2 입력·출력 경로
INSTA_PROC_PATH  = BASE_DIR / 'data' / 'processed' / 'insta_keywords_processed.parquet'
BLOG_PROC_PATH   = BASE_DIR / 'data' / 'processed' / 'blog_keywords_processed.parquet'
TREND_PROC_PATH  = TREND_KW_PATH   # alias
B4_FILTERED_PATH = B4_PATH         # alias
OUT_PRODUCT      = BASE_DIR / 'data' / 'processed' / 'product_master_dataset.parquet'

for label, p in [('insta_keywords_processed', INSTA_PROC_PATH),
                  ('blog_keywords_processed',  BLOG_PROC_PATH),
                  ('trend_keywords_processed', TREND_PROC_PATH),
                  ('B4_filtered',              B4_FILTERED_PATH)]:
    print(f'  {label:30s}: {"OK" if p.exists() else "MISSING"}')


  insta_keywords_processed      : OK
  blog_keywords_processed       : OK
  trend_keywords_processed      : OK
  B4_filtered                   : OK


---
# Part 2: POS Feature Engineering

**Phase 1A**: 상품별 첫 판매일 계산  
**Phase 1B**: 첫 판매일 기준 30일 매출 집계  
**Phase 1C**: B5 행사 데이터 — 출시 첫 달 프로모션 여부  
**Phase 1D**: 최종 병합 및 저장  

## Phase 0: NPD 상품 목록 로드

In [2]:
b4 = pd.read_parquet(B4_PATH)
b4_npd = b4[b4['is_npd']].copy()
npd_ids = set(b4_npd['ITEM_CD'].tolist())

print(f'전체 B4 상품: {len(b4):,}')
print(f'NPD 상품 수 (is_npd=True): {len(npd_ids):,}')
b4_npd[['ITEM_CD', 'ITEM_NM', 'ITEM_LRDV_NM', 'ITEM_MDDV_NM', '생존여부']].head(5)

전체 B4 상품: 49,957
NPD 상품 수 (is_npd=True): 3,128


,ITEM_CD,ITEM_NM,ITEM_LRDV_NM,ITEM_MDDV_NM,생존여부
1003,053047,존슨빌)폴리쉬소시지108g,즉석 식품,즉석조리,생존
1050,053197,요기요)만쿠만구치킨,즉석 식품,즉석치킨,생존
1357,014173,예약)한끼연구소 간장불고기정식,미반,도시락,생존
1380,014248,APP예약)더꽉찬불고기&참치김밥,미반,김밥,생존
1743,013268,롯데)참치마요네즈삼각김밥,미반,삼각김밥,생존


## Phase 1A: 첫 판매일 계산

POS 전체(73M행) 중 NPD 상품만 lazy scan → `영업일자` 최솟값 = 첫 판매일.

In [ ]:
print('POS 로딩 중... (NPD 상품 필터 적용)')

pos_npd = (
    pl.scan_parquet(str(POS_PATH))
    .filter(pl.col('상품코드').is_in(list(npd_ids)))
    .select(['상품코드', '영업일자', '매출수량', '매출금액', '거래_고유키'])
    .collect()
)

print(f'NPD 해당 POS 행: {len(pos_npd):,}')
print(f'POS 매출 있는 NPD 상품: {pos_npd["상품코드"].n_unique():,}개')

# 영업일자 int(YYYYMMDD) → date
pos_npd = pos_npd.with_columns(
    pl.col('영업일자').cast(pl.Utf8).str.to_date(format='%Y%m%d').alias('영업일자_dt')
)

# 상품별 첫 판매일
first_sale = (
    pos_npd
    .group_by('상품코드')
    .agg(pl.col('영업일자_dt').min().alias('첫판매일'))
)

print(f'\n첫 판매일 범위: {first_sale["첫판매일"].min()} ~ {first_sale["첫판매일"].max()}')
first_sale.head(5)

POS 로딩 중... (NPD 상품 필터 적용)
NPD 해당 POS 행: 8,825,021
POS 매출 있는 NPD 상품: 3,128개

첫 판매일 범위: 2025-01-15 ~ 2025-12-31


상품코드,첫판매일
str,date
"""837041""",2025-11-08
"""129816""",2025-12-11
"""128335""",2025-10-31
"""831289""",2025-01-26
"""129511""",2025-11-17


## Phase 1B: 30일 매출 집계

| 컬럼 | 설명 |
|------|------|
| `sales_30d_qty` | 30일 총 매출 수량 |
| `sales_30d_amt` | 30일 총 매출 금액 |
| `sales_days_observed` | 실제 매출 발생일 수 |
| `daily_velocity` | `sales_30d_qty / 30` |

In [ ]:
pos_with_first = pos_npd.join(first_sale, on='상품코드', how='left')

pos_30d = pos_with_first.filter(
    (pl.col('영업일자_dt') >= pl.col('첫판매일')) &
    (pl.col('영업일자_dt') <= pl.col('첫판매일') + pl.duration(days=29))
)

print(f'30일 윈도우 내 POS 행: {len(pos_30d):,}')

sales_agg = (
    pos_30d
    .group_by('상품코드')
    .agg([
        pl.col('매출수량').sum().alias('sales_30d_qty'),
        pl.col('매출금액').sum().alias('sales_30d_amt'),
        pl.col('영업일자_dt').n_unique().alias('sales_days_observed'),
    ])
    .with_columns(
        (pl.col('sales_30d_qty') / 30.0).alias('daily_velocity')
    )
)

print(f'집계 완료: {len(sales_agg)}개 상품')
print(sales_agg.describe())

30일 윈도우 내 POS 행: 2,475,624
집계 완료: 3128개 상품
shape: (9, 6)
┌────────────┬──────────┬───────────────┬───────────────┬─────────────────────┬────────────────┐
│ statistic  ┆ 상품코드 ┆ sales_30d_qty ┆ sales_30d_amt ┆ sales_days_observed ┆ daily_velocity │
│ ---        ┆ ---      ┆ ---           ┆ ---           ┆ ---                 ┆ ---            │
│ str        ┆ str      ┆ f64           ┆ f64           ┆ f64                 ┆ f64            │
╞════════════╪══════════╪═══════════════╪═══════════════╪═════════════════════╪════════════════╡
│ count      ┆ 3128     ┆ 3128.0        ┆ 3128.0        ┆ 3128.0              ┆ 3128.0         │
│ null_count ┆ 0        ┆ 0.0           ┆ 0.0           ┆ 0.0                 ┆ 0.0            │
│ mean       ┆ null     ┆ 828.31427     ┆ 1.6920e6      ┆ 16.319054           ┆ 27.610477      │
│ std        ┆ null     ┆ 1879.250366   ┆ 4.6953088e7   ┆ 13.023675           ┆ 62.641682      │
│ min        ┆ 012928   ┆ -200.0        ┆ -2.6000e9     ┆ 1.0             

## Phase 1C: B5 행사 — 출시 첫 달 프로모션 여부

겹침 조건: `행사개시일 ≤ 첫판매일+30d AND 행사종료일 ≥ 첫판매일`

In [5]:
b5 = pd.read_parquet(B5_PATH)
print(f'B5 shape: {b5.shape}, 컬럼: {list(b5.columns)}')
b5.head(3)

B5 shape: (59250, 15), 컬럼: ['행사기준', '행사정보코드', '행사코드', '행사명', '행사개시일', '행사종료일', '대분류', '상품코드', '상품명', '행사원가', '행사매가', '할인매가', '행사형태', '행사유형', '정산방식']


,행사기준,행사정보코드,행사코드,행사명,행사개시일,행사종료일,대분류,상품코드,상품명,행사원가,행사매가,할인매가,행사형태,행사유형,정산방식
0,12월 세븐앱 구독행사,153783,635467,202512 세븐카페 구독행사,2025-12-01,2026-01-31,즉석음료,51495,T)아이스바닐라라떼_13oz,0.0,2100,1470.0,NaN,0301 : 구독행사,03 : 사후정산(매가)
1,12월 세븐앱 구독행사,153783,635467,202512 세븐카페 구독행사,2025-12-01,2026-01-31,즉석음료,51689,엔제)타임커피카페머그 16,0.0,13000,9100.0,NaN,0301 : 구독행사,03 : 사후정산(매가)
2,12월 세븐앱 구독행사,153783,635467,202512 세븐카페 구독행사,2025-12-01,2026-01-31,즉석음료,51704,엔제)커피프레스,0.0,27000,18900.0,NaN,0301 : 구독행사,03 : 사후정산(매가)


In [6]:
first_sale_pd = first_sale.to_pandas()
first_sale_pd['첫판매일']  = pd.to_datetime(first_sale_pd['첫판매일'])
first_sale_pd['window_end'] = first_sale_pd['첫판매일'] + pd.Timedelta(days=30)

b5_npd = b5[b5['상품코드'].isin(npd_ids)].copy()
b5_npd['행사개시일'] = pd.to_datetime(b5_npd['행사개시일'])
b5_npd['행사종료일'] = pd.to_datetime(b5_npd['행사종료일'])
print(f'B5 NPD 해당 행: {len(b5_npd):,} / {b5_npd["상품코드"].nunique()}개 상품')

b5_merged = b5_npd.merge(first_sale_pd[['상품코드', '첫판매일', 'window_end']], on='상품코드', how='inner')

b5_overlap = b5_merged[
    (b5_merged['행사개시일'] <= b5_merged['window_end']) &
    (b5_merged['행사종료일'] >= b5_merged['첫판매일'])
].copy()

print(f'첫 달 프로모션 겹침: {len(b5_overlap):,}행 / {b5_overlap["상품코드"].nunique():,}개 상품')

B5 NPD 해당 행: 11,264 / 1354개 상품
첫 달 프로모션 겹침: 3,501행 / 1,090개 상품


In [7]:
def join_unique(s):
    return '|'.join(sorted(s.dropna().astype(str).unique()))

promo_agg = (
    b5_overlap
    .groupby('상품코드', as_index=False)
    .agg(
        promo_count_30d      = ('행사명',   'count'),
        promo_names_30d      = ('행사명',   join_unique),
        promo_types_30d      = ('행사형태', join_unique),
        promo_categories_30d = ('행사유형', join_unique),
    )
)
promo_agg['has_promo_30d'] = True

print(f'프로모션 집계: {len(promo_agg)}개 상품')
promo_agg.head(5)

프로모션 집계: 1090개 상품


,상품코드,promo_count_30d,promo_names_30d,promo_types_30d,promo_categories_30d,has_promo_30d
0,105700,3,"202503 샌드위치,햄버거 구독행사|202503 샌드위치,햄버거 구독행사(핫딜)",,0301 : 구독행사,True
1,105745,6,202505 와인 구독행사|25년 5월 이달의 와인 삼성 20%|25년 5월 이달의...,,0203 : 번들할인|0301 : 구독행사,True
2,105748,6,202503 와인 구독행사|이달의 와인 하나카드 20% 할인|이달의와인 네이버페이 ...,,0203 : 번들할인|0301 : 구독행사,True
3,105947,3,"202503 삼각김밥,김밥 구독행사|202503 삼각김밥,김밥 구독행사(핫딜)",,0301 : 구독행사,True
4,106024,3,"202503 삼각김밥,김밥 구독행사|202503 삼각김밥,김밥 구독행사(핫딜)",,0301 : 구독행사,True


## Phase 1D: 최종 병합 및 저장

In [8]:
df_base = b4_npd[['ITEM_CD', 'ITEM_NM', 'ITEM_LRDV_NM', 'ITEM_MDDV_NM', 'ITEM_SMDV_NM', '생존여부']].copy()

df_base = df_base.merge(first_sale_pd[['상품코드', '첫판매일']],
                        left_on='ITEM_CD', right_on='상품코드', how='left').drop(columns=['상품코드'])

sales_agg_pd = sales_agg.to_pandas()
df_base = df_base.merge(sales_agg_pd, left_on='ITEM_CD', right_on='상품코드', how='left').drop(columns=['상품코드'])

df_base = df_base.merge(promo_agg, left_on='ITEM_CD', right_on='상품코드', how='left').drop(columns=['상품코드'])

# 결측 채우기
df_base['has_promo_30d']   = df_base['has_promo_30d'].fillna(False)
df_base['promo_count_30d'] = df_base['promo_count_30d'].fillna(0).astype(int)
for col in ['sales_30d_qty', 'sales_30d_amt', 'sales_days_observed', 'daily_velocity']:
    df_base[col] = df_base[col].fillna(0.0)
for col in ['promo_names_30d', 'promo_types_30d', 'promo_categories_30d']:
    df_base[col] = df_base[col].fillna('')

print(f'최종 shape: {df_base.shape}')
print(f'컬럼: {list(df_base.columns)}')
print()
has_pos   = df_base['sales_30d_qty'].gt(0).sum()
has_promo = df_base['has_promo_30d'].sum()
print(f'  POS 30일 매출 있는 상품  : {has_pos:,}개  ({has_pos/len(df_base)*100:.1f}%)')
print(f'  첫 달 프로모션 있는 상품 : {has_promo:,}개  ({has_promo/len(df_base)*100:.1f}%)')
df_base.head(5)

최종 shape: (3128, 16)
컬럼: ['ITEM_CD', 'ITEM_NM', 'ITEM_LRDV_NM', 'ITEM_MDDV_NM', 'ITEM_SMDV_NM', '생존여부', '첫판매일', 'sales_30d_qty', 'sales_30d_amt', 'sales_days_observed', 'daily_velocity', 'promo_count_30d', 'promo_names_30d', 'promo_types_30d', 'promo_categories_30d', 'has_promo_30d']

  POS 30일 매출 있는 상품  : 3,006개  (96.1%)
  첫 달 프로모션 있는 상품 : 1,090개  (34.8%)


,ITEM_CD,ITEM_NM,ITEM_LRDV_NM,ITEM_MDDV_NM,ITEM_SMDV_NM,생존여부,첫판매일,sales_30d_qty,sales_30d_amt,sales_days_observed,daily_velocity,promo_count_30d,promo_names_30d,promo_types_30d,promo_categories_30d,has_promo_30d
0,053047,존슨빌)폴리쉬소시지108g,즉석 식품,즉석조리,빅바이트,생존,2025-02-23,1.0,2200.0,1,0.033333,0,,,,False
1,053197,요기요)만쿠만구치킨,즉석 식품,즉석치킨,세트류,생존,2025-04-14,1.0,8720.0,1,0.033333,0,,,,False
2,014173,예약)한끼연구소 간장불고기정식,미반,도시락,기타,생존,2025-05-30,40.0,394666.0,6,1.333333,0,,,,False
3,014248,APP예약)더꽉찬불고기&참치김밥,미반,김밥,말이김밥,생존,2025-04-22,3.0,8700.0,1,0.100000,0,,,,False
4,013268,롯데)참치마요네즈삼각김밥,미반,삼각김밥,삼각김밥,생존,2025-03-29,1.0,1000.0,2,0.033333,0,,,,False


In [9]:
# ── 음수 순매출 제품 제거 (반품초과 / 취소 데이터) ─────────────────
import re as _re
from pathlib import Path as _Path

def _nid(x):
    try: return str(int(float(x)))
    except: return str(x)

_neg_mask = df_base['sales_30d_amt'] <= 0
_neg_items = df_base[_neg_mask][['ITEM_CD','ITEM_NM','ITEM_MDDV_NM','sales_30d_amt','sales_30d_qty']].copy()
print(f'[음수 매출 제거] {_neg_mask.sum()}개')
if len(_neg_items) > 0:
    print(_neg_items.to_string(index=False))
df_base = df_base[~_neg_mask].reset_index(drop=True)

# ── product_name_review_final / blog_match_fail_review 제거 대상 ──
_PROC_DIR = _Path(B4_PATH).parent
_REV_PATH      = _PROC_DIR / 'product_name_review_final.xlsx'
_BLOG_FAIL_PATH = _PROC_DIR / 'blog_match_fail_review.xlsx'

_SPEC_RE_01 = _re.compile(r'\d[\d.]*\s*[gGmMlLkK개입봉팩Pp].*$', _re.IGNORECASE)
def _norm01(s):
    s = _re.sub(r'^[^)]+\)', '', str(s)).strip()
    s = _SPEC_RE_01.sub('', s)
    s = _re.sub(r'[\s!&\(\)\[\].·ㆍ★☆▶▷]', '', s).lower()
    return s.strip()

_excl_names = set()
if _REV_PATH.exists():
    _rv = pd.read_excel(_REV_PATH)
    _excl_names |= set(
        _rv[_rv['수정명'].astype(str).str.strip().str.upper()=='O']['원본명']
        .dropna().astype(str).str.strip()
    )
if _BLOG_FAIL_PATH.exists():
    _bf = pd.read_excel(_BLOG_FAIL_PATH)
    _ecol = next((c for c in _bf.columns if '제거' in c), None)
    if _ecol:
        _excl_names |= set(
            _bf[_bf[_ecol].astype(str).str.strip().str.upper()=='O']['블로그_상품명']
            .dropna().astype(str).str.strip()
        )
print(f'[review 제거 대상] 이름 합산: {len(_excl_names)}개')

_nm2id = {str(r.ITEM_NM).strip(): _nid(r.ITEM_CD) for _, r in df_base.iterrows()}
_no2id = {_norm01(r.ITEM_NM): _nid(r.ITEM_CD) for _, r in df_base.iterrows()}
_excl_ids = {_nm2id.get(n) or _no2id.get(_norm01(n)) for n in _excl_names} - {None}

_excl_mask = df_base['ITEM_CD'].apply(_nid).isin(_excl_ids)
print(f'  → ITEM_CD 매핑: {len(_excl_ids)}개 / pos_product_features 제거: {_excl_mask.sum()}개')
df_base = df_base[~_excl_mask].reset_index(drop=True)

# ── B4_filtered is_npd 동기화 (음수매출 + review 제거 통합) ────────
_all_invalid = set(_neg_items['ITEM_CD'].apply(_nid)) | _excl_ids
if _all_invalid:
    _b4 = pd.read_parquet(B4_PATH)
    _b4n = _b4['ITEM_CD'].apply(_nid)
    _upd = _b4n.isin(_all_invalid).sum()
    _b4.loc[_b4n.isin(_all_invalid), 'is_npd'] = False
    _b4.to_parquet(B4_PATH, index=False)
    print(f'B4_filtered is_npd → False 처리: {_upd}개')

# ── 저장 ──────────────────────────────────────────────────────────────
df_base.to_parquet(OUT_PATH, index=False, engine='pyarrow')
print(f'Parquet 저장: {OUT_PATH}')
df_base.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
print(f'CSV    저장: {OUT_CSV}')
print(f'shape: {df_base.shape}')


[음수 매출 제거] 122개
ITEM_CD              ITEM_NM ITEM_MDDV_NM  sales_30d_amt  sales_30d_qty
 061901   빙그레)토핑요플레(쿠앤크)120g         요구르트   0.000000e+00            0.0
 062200       서울)강릉커피라떼250ml          컵커피   0.000000e+00            0.0
 062513       매일)피크닉청포도200ml         냉장주스   0.000000e+00            0.0
 073320      풀무원)나주식수육곰탕450g        냉장간편식   0.000000e+00            0.0
 104713           이멕스)봉봉(초코)          초콜릿   0.000000e+00            0.0
 119326        롯데)빈츠카페모카102g         비스킷류  -1.680000e+04           -6.0
 118216       롯데)ABC초코쿠키152g         비스킷류   0.000000e+00            0.0
 132718           웅진)만한대찬우육면          용기면   0.000000e+00            0.0
 192514          윌리엄)글렌피딕30년           양주   0.000000e+00            0.0
 200357     페르)로얄살루트21년700ml           양주   0.000000e+00            0.0
 191004     페르)앱솔루트라즈베리375ml           양주   0.000000e+00            0.0
 200440        스카치블루21년500ml           양주   0.000000e+00            0.0
 200929     칠성)데일리C레몬워터500ml       기능성드링크   0.00